# RAD-MOT trên Colab / Kaggle

- Phần 1–3 chỉ cần **CPU** (Runtime → Change runtime type → CPU). Không tốn giờ GPU.
- Phần 4 cần **GPU**. Colab miễn phí (T4) chạy được nhưng chậm hơn GPU có bfloat16 (L4/A100). Đo tốc độ trước bằng `smoke_sam3.py`, rồi ước lượng giờ GPU bằng `estimate_gpu_time.py`.
- Dữ liệu (~14,8 GB) tải vào ổ tạm `/content` của phiên. **Không** tải vào Google Drive (Drive miễn phí chỉ 15 GB). Kết quả lưu vào Drive.

Trước khi chạy: chép thư mục `rad-mot` lên Google Drive, ví dụ `MyDrive/rad-mot`.

## 1. Chuẩn bị

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
CODE = '/content/drive/MyDrive/rad-mot'    # mã nguồn (nhỏ)
WORK = '/content/drive/MyDrive/rad-work'   # kết quả (nhỏ)
DS   = '/content/datasets/FastTracker'     # dữ liệu: ổ tạm của phiên, mất khi tắt phiên
os.makedirs(WORK, exist_ok=True)
%cd $CODE
!pip -q install -r requirements.txt huggingface_hub
!python -m pytest -q

## 2. Tải GT của FastTracker (vài MB) và thống kê lớp

Bộ dữ liệu gồm 12 tệp zip (mỗi sequence một tệp, 0,5–1,6 GB). Script dưới chỉ đọc `gt/gt.txt` và `seqinfo.ini` bên trong từng zip qua HTTP Range, không tải cả zip. Nếu máy chủ không cho đọc theo đoạn, script tự tải từng zip, lấy 2 tệp đó rồi xoá zip.

In [ ]:
!python scripts/download_fasttracker.py --out $DS --mode gt
!find $DS -maxdepth 3 | head -40

In [ ]:
DATA = f'{DS}/train'
!python scripts/dataset_stats.py --data $DATA --out $WORK/outputs/stats
# Mã lớp theo GT của FastTracker (rad/classes.py): 8 = motorbike, 9 = ignore_region (vùng bỏ qua, bị loại ở mọi bước).
# per_sequence.csv không tính vùng bỏ qua; class_totals.csv liệt kê đủ mọi lớp, share_of_objects không tính lớp 9.

## 3. Pilot H1/H2 và Risk-Aware Dispatch (CPU, VOS oracle)

In [ ]:
!python scripts/run_pilot.py --data $DATA --dets oracle --backend oracle --out $WORK/outputs/pilot
!python scripts/eval_trackeval.py --data $DATA --results $WORK/outputs/pilot --name bytetrack.txt
!python scripts/eval_trackeval.py --data $DATA --results $WORK/outputs/pilot --name smp_oracle.txt

In [ ]:
from IPython.display import Image, Markdown, display
for p in sorted(glob.glob(f'{WORK}/outputs/pilot/*.png')): display(Image(p))
display(Markdown(open(f'{WORK}/outputs/pilot/pilot_report.md').read()))

### Độ nhạy witness (CPU)
Witness windows chiếm bao nhiêu chi phí và bao nhiêu lỗi được sửa. Mỗi lần chạy mất thời gian xấp xỉ cell pilot ở trên.

In [ ]:
!python scripts/run_pilot.py --data $DATA --witness-iou 0.3 --out $WORK/outputs/pilot_wit03
!python scripts/run_pilot.py --data $DATA --no-witness --out $WORK/outputs/pilot_nowit
import pandas as pd
cols = ['windows', 'windows_witness', 'vos_frames', 'base_idsw', 'smp_idsw', 'switches_fixed', 'switches_introduced']
runs = [('witness IoU > 0', 'pilot'), ('witness IoU > 0.3', 'pilot_wit03'), ('no witness', 'pilot_nowit')]
display(pd.DataFrame({name: pd.read_csv(f'{WORK}/outputs/{d}/per_sequence.csv')[cols].sum() for name, d in runs}).T)

In [ ]:
!python scripts/run_dispatch.py --data $DATA --dets oracle --backend oracle --label y_risk --out $WORK/outputs/dispatch
for p in sorted(glob.glob(f'{WORK}/outputs/dispatch/*.png')): display(Image(p))

## 4. GPU (đổi runtime sang T4 / GPU)

Cần quyền tải checkpoint SAM 3: xin tại https://huggingface.co/facebook/sam3 rồi đăng nhập ở ô dưới. Khi đổi runtime, chạy lại ô ở phần 1 và 2 (ổ `/content` bị xoá).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Ảnh của các sequence cần dùng (mỗi sequence 0,5–1,6 GB). Bắt đầu với 1–2 sequence, ví dụ:
!python scripts/download_fasttracker.py --out $DS --mode full --seqs task_tunnel task_night_far_objects
# Tải hết 12 sequence (~15 GB): bỏ --seqs
!df -h /content | tail -1

In [ ]:
!git clone -q https://github.com/facebookresearch/sam3 /content/sam3 && pip -q install -e /content/sam3 ultralytics
from huggingface_hub import notebook_login
notebook_login()

### 4a. Đo tốc độ SAM 3 trên GPU này (làm đầu tiên)
Trên T4 thử cả hai chế độ: `auto` (bfloat16 giả lập) và `fp16` (thử nghiệm). Chọn chế độ nhanh hơn **với điều kiện** số khung "mask stays on its GT box" không giảm.

In [ ]:
!python scripts/smoke_sam3.py --data $DATA --frame 100 --objects 5 --len 60 --precision auto
!python scripts/smoke_sam3.py --data $DATA --frame 100 --objects 5 --len 60 --precision fp16

In [ ]:
MS = 300   # điền ms/frame đo được ở trên
!python scripts/estimate_gpu_time.py --ms-per-frame $MS --pilot $WORK/outputs/pilot/per_sequence.csv --sweep $WORK/outputs/dispatch/sweep_summary.csv

### 4b. YOLO và SAM 3 thật (chia nhỏ theo sequence để không mất kết quả khi phiên bị ngắt)
Nếu chọn `fp16` ở 4a, sửa `sam3_backend: precision: fp16` trong `configs/default.yaml`.

In [ ]:
!python scripts/detect_yolo.py --data $DATA --weights yolo11l.pt --imgsz 1280 --half --out $WORK/outputs/dets
!python scripts/run_pilot.py --data $DATA --dets dir --det-dir $WORK/outputs/dets --backend oracle --out $WORK/outputs/pilot_yolo

In [ ]:
SEQS = sorted(os.listdir(DATA))
for s in SEQS[:2]:   # từng sequence một; kết quả ghi vào Drive sau mỗi lần
    !python scripts/run_pilot.py --data $DATA --dets dir --det-dir $WORK/outputs/dets --backend sam3 --seqs $s --out $WORK/outputs/pilot_sam3/$s